# 05a — Audit Data (Read-Only) dan Pembuatan Folds

**Tujuan.** Mengaudit `cleaned_master_dataset.csv` (output Notebook 04) tanpa mengubah apa pun, lalu membuat pembagian fold **per subjek** yang akan dipakai semua model berikutnya (tabular, citra 2D, 3D).

**Prinsip.**
- Read-only: `cleaned_master_dataset.csv` tidak ditulis ulang. Hash SHA-256 dicek sebelum dan sesudah.
- Kolom `label` hanya ada di memori dan di `folds.csv`. Hasil audit per subjek hanya ada di `audit_flags.csv`.
- Tidak ada transformasi yang di-*fit*. Statistik di notebook ini (IQR, z-score, uji konfounding) hanya untuk **menandai dan melaporkan**, bukan untuk mengubah data atau membuat fitur.

| CELL | Isi |
|---|---|
| 1 | CONFIG |
| 2 | Muat data, hash, cek kolom wajib, encoding DX, `label`, file sMRI |
| 3 | Audit integritas: tipe data, duplikasi, mismatch ID, kategori, rentang, outlier |
| 3c | Hash isi file citra, citra identik, `group_id` |
| 4 | Audit missing value (fokus FIQ) |
| 5 | Audit situs |
| 6 | Audit konfounding |
| 7 | Eksklusi, split dev/test (held-out), folds pada dev, daftar pilot |
| 8 | `audit_flags.csv` |
| 9 | `audit_report.md` dan verifikasi akhir (hash) |
| 10 | Subset 20% (200 subjek dev) untuk batch preprocessing dan eksperimen subset |

**Output** (`data/`): `folds.csv` (memuat `label`, `group_id`, `split`), `exclusions.csv`, `pilot_ids.csv`, `subset_20pct.csv` (+ `subset_20pct_rules.md`), `audit_flags.csv` (memuat hash citra), `audit_report.md`, cache `audit_cache/image_hashes.csv`.

**Skema split (v2):** subjek yang terbukti salah pasangan citra dikeluarkan (`exclusions.csv`), lalu ±15% subjek disisihkan sebagai **test held-out** (stratifikasi label × situs, `group_id`, seed 42). Semua fold CV dan LOSO dibangun **hanya dari dev**. Test tidak dipakai sampai evaluasi akhir. Versi lama (5-fold pada seluruh data) diarsipkan di `data/archive/v1_cv_only/`.

**Environment:** VS Code lokal. Notebook dijalankan dari `notebook/`; semua path relatif terhadap root proyek (`notebook/`, `data/`, `dataset/`).

### CELL 1 — CONFIG

**Tujuan:** semua path dan parameter di satu tempat. **Output:** variabel global.

Catatan:
- Dependensi: `pandas numpy scipy scikit-learn nibabel` (`pip install nibabel` jika belum ada).
- `PROJECT_ROOT` dideteksi dari folder kerja: jika notebook dijalankan dari `notebook/`, root = folder induknya.
- `DATA_ROOT` adalah basis `smri_path` (path di CSV relatif terhadap root proyek, mis. `dataset/all_anat_T1w/...`).
- `LOSO_MIN_PER_CLASS = 10` adalah pilihan metodologis (di bawah ~10 subjek per kelas, AUC per situs sangat tidak stabil), bukan standar baku.

In [ ]:
import hashlib
import re
import shutil
import time
import warnings
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import StratifiedGroupKFold
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

# ---------- Path (relatif terhadap root proyek) ----------
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "dataset").exists() and (PROJECT_ROOT.parent / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent             # dijalankan dari notebook/
DATA_ROOT = PROJECT_ROOT                           # basis smri_path

CLEANED_DIR = PROJECT_ROOT / "dataset" / "cleaned_dataset"
CLEANED_PATH = CLEANED_DIR / "cleaned_master_dataset.csv"
QC_PATH = CLEANED_DIR / "smri_qc.csv"
DICTIONARY_PATH = CLEANED_DIR / "data_dictionary.csv"
RAW_PHENO_PATH = PROJECT_ROOT / "dataset" / "fenotipik" / "Phenotypic_V1_0b.csv"
SMRI_DIR = PROJECT_ROOT / "dataset" / "all_anat_T1w"

OUTPUT_DIR = PROJECT_ROOT / "data"
IMAGE_HASH_CACHE = OUTPUT_DIR / "audit_cache" / "image_hashes.csv"   # resume hashing
FOLDS_PATH = OUTPUT_DIR / "folds.csv"
AUDIT_FLAGS_PATH = OUTPUT_DIR / "audit_flags.csv"
AUDIT_REPORT_PATH = OUTPUT_DIR / "audit_report.md"

# ---------- Seed ----------
SEED = 42
np.random.seed(SEED)

# ---------- Skema data ----------
REQUIRED_COLUMNS = ["SUB_ID", "SITE_ID", "DX_GROUP", "AGE_AT_SCAN", "SEX", "FIQ",
                    "HANDEDNESS_CATEGORY", "FIQ_missing", "smri_path"]
EXPECTED_KIND = {"SUB_ID": "integer", "SITE_ID": "string", "DX_GROUP": "integer",
                 "AGE_AT_SCAN": "float", "SEX": "integer", "FIQ": "float",
                 "HANDEDNESS_CATEGORY": "string", "FIQ_missing": "integer", "smri_path": "string"}

# DX_GROUP bisa berupa kode asli ABIDE (1/2) atau sudah di-recode Notebook 04 (1/0).
# Keduanya dipetakan ke label: ASD = 1, TD = 0.
DX_ENCODINGS = {
    "ABIDE asli (1=ASD, 2=TD)": {1: 1, 2: 0},
    "recode Notebook 04 (1=ASD, 0=TD)": {1: 1, 0: 0},
}
RAW_DX_TO_LABEL = {1: 1, 2: 0}                   # kode di Phenotypic_V1_0b.csv
LABEL_NAMES = {1: "ASD", 0: "TD"}

SEX_VALID = {1, 2}                                # 1 = laki-laki, 2 = perempuan
HANDEDNESS_VALID = {"R", "L", "Ambi"}
AGE_VALID_RANGE = (5, 70)                         # sama dengan Notebook 04 (tahun)
FIQ_VALID_RANGE = (40, 160)                       # sama dengan Notebook 04
RAW_MISSING_SENTINELS = {"-9999", "-9999.0"}
SMRI_ID_PATTERN = re.compile(r"sub-0*(\d+)_T1w\.nii(?:\.gz)?$")
REQUIRE_SMRI_ON_DISK = True                       # True: notebook berhenti jika ada citra hilang
# group_id dibentuk dari hash voxel (data terdekompresi + shape + dtype + affine), bukan hash file .nii.gz:
# dua file gzip bisa berbeda byte (header gzip: nama/waktu) walau isi citranya identik.
GROUP_HASH = "voxel"                              # "voxel" (disarankan) | "file"
HASH_CACHE_EVERY = 100                            # simpan cache tiap N subjek

# ---------- Audit ----------
OUTLIER_IQR_K = 1.5
OUTLIER_Z = 3.0                                   # z-score global
OUTLIER_ROBUST_Z = 3.5                            # robust z per situs (median/MAD)
SITE_MISSING_STRUCTURAL = 0.5                     # >= 50% NaN di satu situs = tidak diukur di situs itu
ALPHA = 0.05
SITE_SPLIT_PATTERN = re.compile(r"^(?P<base>.+?)_(?P<idx>\d+)$")
ABIDE_I_STANDARD_SITES = 17

# ---------- Folds ----------
N_FOLDS = 5
REPEAT_SEEDS = [SEED + 1, SEED + 2, SEED + 3]     # repeated_fold_1..3
LOSO_MIN_PER_CLASS = 10                           # perlu dicek (lihat catatan di atas)

# ---------- Split v2: eksklusi + held-out test ----------
SPLIT_SCHEMA_VERSION = "v2_holdout15_devcv"
TEST_FRACTION = 0.15                              # per strata label × SITE_ID, dibulatkan
# Citra identik dengan fenotipik berbeda (satu scan tertaut ke dua subjek). Dikeluarkan hanya jika bukti audit
# (group_id sama, hash voxel sama, fenotipik berbeda) terkonfirmasi di CELL 7.
EXCLUDE_SUB_IDS = [50279, 50286]
EXCLUDE_REASON = ("citra T1w identik (hash voxel sama) tertaut ke dua SUB_ID dengan fenotipik berbeda; "
                  "pemilik fenotipik yang benar tidak dapat ditentukan")
PILOT_N = 20                                      # daftar pilot 05c, diambil dari dev
EXCLUSIONS_PATH = OUTPUT_DIR / "exclusions.csv"
PILOT_IDS_PATH = OUTPUT_DIR / "pilot_ids.csv"
# Subset 20%: hanya dari dev, mencakup semua pilot, stratifikasi fold_strat × SITE_ID × label (proporsional)
SUBSET_N = 200
SUBSET_PATH = OUTPUT_DIR / "subset_20pct.csv"
SUBSET_RULES_PATH = OUTPUT_DIR / "subset_20pct_rules.md"
ARCHIVE_DIR = OUTPUT_DIR / "archive" / "v1_cv_only"
# Output 05a/05b skema lama yang diarsipkan (disalin, tidak dipindah) sebelum ditimpa
ARCHIVE_FILES = ["folds.csv", "audit_flags.csv", "audit_report.md", "pheno_clean.csv", "cleaning_log.csv",
                 "drop_log.csv", "missing_decision.csv", "pheno_verification_summary.md"]

assert SEED == 42 and N_FOLDS >= 2 and SEED not in REPEAT_SEEDS
assert GROUP_HASH in {"voxel", "file"}
assert (PROJECT_ROOT / "dataset").exists(), f"Root proyek tidak ditemukan dari {Path.cwd()}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_HASH_CACHE.parent.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("OUTPUT_DIR   :", OUTPUT_DIR)

### CELL 2 — Muat data, hash, kolom wajib, encoding DX, label, file sMRI

**Tujuan:** memuat data secara read-only, mencatat hash file sumber, memvalidasi kolom wajib, `SUB_ID` unik, encoding `DX_GROUP`, membuat `label` (hanya di memori), dan mengecek keberadaan file sMRI.

**Output:** `df_raw` (tidak pernah diubah), `df` (salinan kerja), `label`, `flags` (tabel flag per subjek), `HASH_BEFORE`.

Catatan: Notebook 04 sudah me-*recode* `DX_GROUP` menjadi 1 = ASD, 0 = TD (lihat `data_dictionary.csv`). Cell ini mendeteksi encoding secara otomatis dan mengonfirmasinya terhadap `Phenotypic_V1_0b.csv` mentah (kode 1/2).

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


FINDINGS = []        # isi audit_report.md
REPORT_TABLES = {}   # tabel yang ikut ditulis ke audit_report.md


def note(section, kind, text):
    """kind: 'temuan' | 'keputusan' | 'tinjau' (perlu ditinjau manual)."""
    assert kind in {"temuan", "keputusan", "tinjau"}
    FINDINGS.append({"bagian": section, "jenis": kind, "teks": text})
    print(f"[{kind.upper()}] {text}")


for p in [CLEANED_PATH, QC_PATH, DICTIONARY_PATH]:
    assert p.exists(), f"File tidak ditemukan: {p}"

HASH_BEFORE = sha256_file(CLEANED_PATH)
print("SHA-256 cleaned_master_dataset.csv (sebelum):", HASH_BEFORE)
FOLDS_HASH_BEFORE = sha256_file(FOLDS_PATH) if FOLDS_PATH.exists() else None
print("SHA-256 folds.csv (sebelum)                 :", FOLDS_HASH_BEFORE or "(belum ada)")
OLD_SCHEMA = FOLDS_PATH.exists() and "split" not in pd.read_csv(FOLDS_PATH, nrows=1).columns
ARCHIVE_MANIFEST = []
if OLD_SCHEMA:
    # Salin (bukan pindah) output skema lama, verifikasi hash; file asli ditimpa baru di CELL 7b/8/9.
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    for name in ARCHIVE_FILES:
        src_p, dst_p = OUTPUT_DIR / name, ARCHIVE_DIR / name
        if not src_p.exists():
            continue
        h_src = sha256_file(src_p)
        if dst_p.exists():
            if sha256_file(dst_p) != h_src:
                raise RuntimeError(f"Arsip {dst_p} sudah ada dengan isi berbeda; tidak ditimpa. Periksa manual.")
        else:
            shutil.copy2(src_p, dst_p)
        assert sha256_file(dst_p) == h_src and dst_p.stat().st_size == src_p.stat().st_size, f"Salinan {name} tidak cocok"
        ARCHIVE_MANIFEST.append({"file": name, "sha256": h_src, "bytes": src_p.stat().st_size})
    pd.DataFrame(ARCHIVE_MANIFEST).to_csv(ARCHIVE_DIR / "manifest.csv", index=False)
    print(f"Skema folds lama terdeteksi (tanpa kolom split): {len(ARCHIVE_MANIFEST)} file disalin ke "
          f"{ARCHIVE_DIR.relative_to(PROJECT_ROOT)} dan diverifikasi (SHA-256 + ukuran).")

df_raw = pd.read_csv(CLEANED_PATH)          # referensi, tidak diubah
df = df_raw.copy()
qc = pd.read_csv(QC_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)
raw_pheno = (pd.read_csv(RAW_PHENO_PATH, dtype=str, keep_default_na=False)
             if RAW_PHENO_PATH.exists() else None)
if raw_pheno is not None:
    raw_pheno["SUB_ID"] = raw_pheno["SUB_ID"].str.strip()
    raw_pheno = raw_pheno.set_index("SUB_ID")
else:
    note("Data", "tinjau", f"{RAW_PHENO_PATH.name} tidak ditemukan; cross-check ke data mentah dilewati.")

# 1) Kolom wajib dan SUB_ID unik
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
assert not missing_cols, f"Kolom wajib hilang: {missing_cols}"
assert df["SUB_ID"].notna().all() and df["SUB_ID"].is_unique, "SUB_ID kosong atau duplikat"
extra_cols = [c for c in df.columns if c not in REQUIRED_COLUMNS]
print("Kolom tambahan di luar kolom wajib:", extra_cols or "-")

# 2) Encoding DX_GROUP
dx_values = {int(v) for v in df["DX_GROUP"].dropna().unique()}
assert df["DX_GROUP"].notna().all(), "DX_GROUP mengandung NaN"
assert len(dx_values) == 2, f"DX_GROUP harus punya tepat 2 nilai, ditemukan {dx_values}"
DX_SCHEME = next((name for name, enc in DX_ENCODINGS.items() if dx_values == set(enc)), None)
assert DX_SCHEME is not None, f"Encoding DX_GROUP tidak dikenali: {dx_values}"
label = df["DX_GROUP"].astype(int).map(DX_ENCODINGS[DX_SCHEME]).astype(int)
label.name = "label"
dx_doc = dictionary.loc[dictionary["column"] == "DX_GROUP", "encoding"]
print("Nilai DX_GROUP :", sorted(dx_values), "->", DX_SCHEME)
print("data_dictionary:", dx_doc.iloc[0] if len(dx_doc) else "(tidak ada entri)")
if DX_SCHEME != "ABIDE asli (1=ASD, 2=TD)":
    note("Data", "temuan", f"DX_GROUP di file berisi {sorted(dx_values)}, bukan {{1, 2}}: sudah di-recode "
         f"Notebook 04 ({DX_SCHEME}). label = DX_GROUP; DX_GROUP asli tidak diubah.")

# Konfirmasi arah encoding ke data mentah (mencegah label terbalik)
if raw_pheno is not None:
    raw_dx = df["SUB_ID"].astype(str).map(raw_pheno["DX_GROUP"]).astype(float)
    label_from_raw = raw_dx.map(RAW_DX_TO_LABEL)
    n_dx_unmatched = int(label_from_raw.isna().sum())
    n_dx_conflict = int((label_from_raw.notna() & (label_from_raw != label)).sum())
    print(f"Cross-check label vs data mentah: tidak ditemukan={n_dx_unmatched}, konflik={n_dx_conflict}")
    dx_crosstab = pd.crosstab(df["DX_GROUP"].rename("DX_GROUP bersih"),
                              raw_dx.fillna(-1).astype(int).replace(-1, "TIDAK DITEMUKAN").rename("DX_GROUP mentah"))
    print(f"Crosstab DX_GROUP bersih × mentah ({RAW_PHENO_PATH.name}):")
    display(dx_crosstab)
    REPORT_TABLES["Verifikasi silang DX_GROUP bersih × mentah"] = dx_crosstab
    note("Data", "temuan", f"Verifikasi silang DX_GROUP vs {RAW_PHENO_PATH.name}: "
         f"{'LOLOS' if n_dx_conflict == 0 else 'GAGAL'} (tidak ditemukan={n_dx_unmatched}, konflik={n_dx_conflict}).")
    assert n_dx_conflict == 0, "Label tidak konsisten dengan DX_GROUP mentah. Encoding mungkin terbalik."

# 3) Tabel flag per subjek (akan ditulis ke audit_flags.csv di CELL 8)
flags = df[["SUB_ID", "SITE_ID"]].copy()

# 4) File sMRI di disk
smri_abs = df["smri_path"].map(lambda p: DATA_ROOT / Path(str(p)))
flags["flag_smri_missing"] = ~smri_abs.map(Path.exists)
n_smri_missing = int(flags["flag_smri_missing"].sum())
print(f"File sMRI tidak ditemukan: {n_smri_missing} / {len(df)}")
if n_smri_missing:
    note("Data", "tinjau", f"{n_smri_missing} file sMRI tidak ditemukan di {SMRI_DIR}.")

### CELL 2b — Verifikasi CELL 2

In [ ]:
assert len(df) == len(df_raw) and df.equals(df_raw)
assert label.isin([0, 1]).all() and len(label) == len(df)
assert "label" not in df.columns, "label tidak boleh masuk ke df / file sumber"
if REQUIRE_SMRI_ON_DISK:
    assert n_smri_missing == 0, f"{n_smri_missing} file sMRI hilang (set REQUIRE_SMRI_ON_DISK=False untuk audit tanpa citra)"

print(f"Baris: {len(df)} | Kolom: {df.shape[1]} | Situs: {df['SITE_ID'].nunique()}")
print("Label:", label.map(LABEL_NAMES).value_counts().to_dict())
print("File sMRI hilang:", n_smri_missing)
print("\nDistribusi kelas per situs:")
dist_site = pd.crosstab(df["SITE_ID"], label.map(LABEL_NAMES), margins=True, margins_name="TOTAL")
display(dist_site)

### CELL 3 — Audit integritas

**Tujuan:** memeriksa tipe data, duplikasi baris/`SUB_ID`/path, mismatch ID antara tabel, tabel QC, dan file citra, konsistensi kategori (`SEX`, `HANDEDNESS_CATEGORY`), rentang dan satuan `AGE_AT_SCAN`/`FIQ`, serta konsistensi dengan data mentah. Outlier hanya **ditandai** (IQR, z-score global, robust z per situs), tidak dibuang.

**Output:** tabel tipe data, ringkasan duplikasi/mismatch, tabel outlier, kolom flag baru di `flags`.

In [ ]:
def dtype_kind(s):
    if pd.api.types.is_bool_dtype(s):
        return "bool"
    if pd.api.types.is_integer_dtype(s):
        return "integer"
    if pd.api.types.is_float_dtype(s):
        return "float"
    return "string"


# 1) Tipe data
dtype_rows = []
for col in df.columns:
    s = df[col]
    kind = dtype_kind(s)
    expected = EXPECTED_KIND.get(col, "-")
    ok = (kind == expected) or expected == "-" or (expected == "float" and kind == "integer")
    non_integer = None
    if kind == "float":
        nn = s.dropna()
        non_integer = int((nn != nn.round()).sum())
    dtype_rows.append({"kolom": col, "dtype": str(s.dtype), "jenis": kind, "diharapkan": expected,
                       "ok": ok, "n_unik": s.nunique(dropna=True), "n_NaN": int(s.isna().sum()),
                       "n_desimal_non_bulat": non_integer, "contoh": list(s.dropna().unique()[:3])})
dtype_tbl = pd.DataFrame(dtype_rows)
display(dtype_tbl)
REPORT_TABLES["Tipe data"] = dtype_tbl.drop(columns="contoh")
bad_dtype = dtype_tbl.loc[~dtype_tbl["ok"], "kolom"].tolist()
if bad_dtype:
    note("Integritas", "tinjau", f"Tipe data tidak sesuai harapan: {bad_dtype}")
fiq_frac = dtype_tbl.loc[dtype_tbl["kolom"] == "FIQ", "n_desimal_non_bulat"].iloc[0]
if fiq_frac:
    frac_sites = df.loc[df["FIQ"].notna() & (df["FIQ"] != df["FIQ"].round()), "SITE_ID"].value_counts().to_dict()
    note("Integritas", "temuan", f"{int(fiq_frac)} nilai FIQ tidak bulat, terkonsentrasi di {frac_sites}. Kemungkinan "
         "berasal dari skor sumber situs (dibiarkan apa adanya; perlu dicek ke dokumentasi situs).")

# 2) Duplikasi
n_dup_row = int(df.duplicated().sum())
n_dup_sub = int(df["SUB_ID"].duplicated().sum())
n_dup_path = int(df["smri_path"].duplicated().sum())
content_cols = [c for c in df.columns if c not in ("SUB_ID", "smri_path")]
dup_content = df[df.duplicated(subset=content_cols, keep=False)].sort_values(content_cols)
print(f"Baris duplikat penuh: {n_dup_row} | SUB_ID duplikat: {n_dup_sub} | smri_path duplikat: {n_dup_path}")
print(f"Baris dengan isi fenotipik+situs identik tetapi SUB_ID berbeda: {len(dup_content)}")
if len(dup_content):
    display(dup_content)
    note("Integritas", "temuan", f"{len(dup_content)} baris memiliki isi fenotipik identik dengan SUB_ID berbeda. "
         "Ini wajar pada data diskret (umur bulat, FIQ NaN); citranya dicek di CELL 3c.")
flags["flag_dup_content"] = df.index.isin(dup_content.index)

# 3) Mismatch ID: tabel vs nama file citra
parsed = df["smri_path"].astype(str).str.extract(SMRI_ID_PATTERN.pattern)[0]
flags["flag_id_mismatch"] = parsed.isna() | (parsed.astype(float) != df["SUB_ID"].astype(float))
n_id_mismatch = int(flags["flag_id_mismatch"].sum())
print(f"Mismatch SUB_ID vs nama file citra: {n_id_mismatch}")

# File citra di folder yang tidak ada di tabel (diharapkan = subjek EXCLUDE Notebook 04)
qc_excluded = set(qc.loc[~qc["included_in_final"].astype(bool), "SUB_ID"].astype(int))
if SMRI_DIR.exists():
    disk_ids = {int(m.group(1)) for f in SMRI_DIR.iterdir() if (m := SMRI_ID_PATTERN.search(f.name))}
    orphan = disk_ids - set(df["SUB_ID"])
    unexplained = orphan - qc_excluded
    print(f"File citra di disk: {len(disk_ids)} | tidak ada di tabel: {len(orphan)} "
          f"(dijelaskan oleh EXCLUDE: {len(orphan & qc_excluded)}, tidak dijelaskan: {len(unexplained)})")
    if unexplained:
        note("Integritas", "tinjau", f"{len(unexplained)} file citra tidak ada di tabel dan bukan EXCLUDE: "
             f"{sorted(unexplained)[:10]}")
else:
    print(f"Folder citra tidak ada ({SMRI_DIR}); cek file yatim dilewati.")

# Tabel vs smri_qc.csv
qc_in = qc[qc["included_in_final"].astype(bool)].drop_duplicates("SUB_ID").set_index("SUB_ID")
only_table = set(df["SUB_ID"]) - set(qc_in.index)
only_qc = set(qc_in.index) - set(df["SUB_ID"])
print(f"SUB_ID hanya di tabel: {len(only_table)} | hanya di smri_qc (included): {len(only_qc)}")
flags["qc_decision"] = df["SUB_ID"].map(qc_in["qc_decision"])
flags["qc_reason"] = df["SUB_ID"].map(qc_in["qc_reason"])
site_conflict = int((df["SUB_ID"].map(qc_in["SITE_ID"]) != df["SITE_ID"]).sum())
print(f"SITE_ID berbeda antara tabel dan smri_qc: {site_conflict}")
print("Status QC:", flags["qc_decision"].value_counts(dropna=False).to_dict())

# 4) Konsistensi kategori
def spelling_variants(series):
    s = series.dropna().astype(str)
    key = s.str.strip().str.lower()
    groups = s.groupby(key).unique()
    return {k: list(v) for k, v in groups.items() if len(v) > 1}

sex_vals = set(df["SEX"].dropna().astype(int).unique())
hand_vals = set(df["HANDEDNESS_CATEGORY"].dropna().unique())
print("\nSEX:", df["SEX"].value_counts(dropna=False).to_dict())
print("HANDEDNESS_CATEGORY:", df["HANDEDNESS_CATEGORY"].value_counts(dropna=False).to_dict())
hand_variants = spelling_variants(df["HANDEDNESS_CATEGORY"])
print("Varian ejaan HANDEDNESS (beda kapital/spasi):", hand_variants or "-")
if sex_vals - SEX_VALID:
    note("Integritas", "tinjau", f"Kode SEX di luar {SEX_VALID}: {sex_vals - SEX_VALID}")
if (hand_vals - HANDEDNESS_VALID) or hand_variants:
    note("Integritas", "tinjau", f"Kategori handedness tidak standar: {hand_vals - HANDEDNESS_VALID}, varian: {hand_variants}")

# 5) Rentang dan satuan
range_rows = []
for col, (lo, hi) in [("AGE_AT_SCAN", AGE_VALID_RANGE), ("FIQ", FIQ_VALID_RANGE)]:
    s = df[col]
    range_rows.append({"kolom": col, "min": s.min(), "q1": s.quantile(.25), "median": s.median(),
                       "q3": s.quantile(.75), "max": s.max(), "rentang_valid": f"[{lo}, {hi}]",
                       "n_di_luar_rentang": int(((s < lo) | (s > hi)).sum()), "n_NaN": int(s.isna().sum())})
range_tbl = pd.DataFrame(range_rows)
display(range_tbl)
REPORT_TABLES["Rentang AGE dan FIQ"] = range_tbl
# Satuan umur: bila tahun, nilai maksimum < 100; bila bulan, median ~ 100-300
age_unit = "tahun" if df["AGE_AT_SCAN"].max() < 100 else "kemungkinan bulan"
print("Satuan AGE_AT_SCAN terdeteksi:", age_unit)
if age_unit != "tahun":
    note("Integritas", "tinjau", "AGE_AT_SCAN tampaknya bukan dalam tahun.")

# 6) Konsistensi dengan data mentah (deteksi nilai yang berubah tanpa tercatat)
if raw_pheno is not None:
    def raw_numeric(col):
        v = df["SUB_ID"].astype(str).map(raw_pheno[col].str.strip())
        v = v.where(~v.isin(RAW_MISSING_SENTINELS | {""}))
        return pd.to_numeric(v, errors="coerce")
    raw_cmp = []
    for col in ["AGE_AT_SCAN", "SEX", "FIQ"]:
        r = raw_numeric(col)
        both = r.notna() & df[col].notna()
        diff = both & ~np.isclose(r.fillna(0), df[col].fillna(0), atol=1e-6)
        raw_cmp.append({"kolom": col, "n_dibandingkan": int(both.sum()), "n_beda": int(diff.sum()),
                        "NaN_di_tabel_tapi_ada_di_mentah": int((df[col].isna() & r.notna()).sum())})
    raw_cmp = pd.DataFrame(raw_cmp)
    print("\nCross-check ke Phenotypic_V1_0b.csv:")
    display(raw_cmp)
    if raw_cmp["n_beda"].sum():
        note("Integritas", "tinjau", "Ada nilai yang berbeda dari data mentah: "
             + raw_cmp.loc[raw_cmp["n_beda"] > 0, ["kolom", "n_beda"]].to_dict("records").__str__())

# 7) Outlier (hanya ditandai)
def iqr_flag(s, k):
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    return ((s < q1 - k * iqr) | (s > q3 + k * iqr)).fillna(False)

def z_flag(s, thr):
    z = (s - s.mean()) / s.std(ddof=0)
    return (z.abs() > thr).fillna(False)

def robust_z_by_site(s, site, thr):
    med = s.groupby(site).transform("median")
    mad = (s - med).abs().groupby(site).transform("median") * 1.4826
    rz = (s - med) / mad.replace(0, np.nan)
    return (rz.abs() > thr).fillna(False)

outlier_rows = []
for col in ["AGE_AT_SCAN", "FIQ"]:
    tag = "age" if col == "AGE_AT_SCAN" else "fiq"
    flags[f"outlier_{tag}_iqr"] = iqr_flag(df[col], OUTLIER_IQR_K)
    flags[f"outlier_{tag}_z"] = z_flag(df[col], OUTLIER_Z)
    flags[f"outlier_{tag}_site_rz"] = robust_z_by_site(df[col], df["SITE_ID"], OUTLIER_ROBUST_Z)
    outlier_rows.append({"kolom": col,
                         f"IQR (k={OUTLIER_IQR_K})": int(flags[f"outlier_{tag}_iqr"].sum()),
                         f"|z|>{OUTLIER_Z}": int(flags[f"outlier_{tag}_z"].sum()),
                         f"robust |z|>{OUTLIER_ROBUST_Z} per situs": int(flags[f"outlier_{tag}_site_rz"].sum())})
outlier_tbl = pd.DataFrame(outlier_rows)
display(outlier_tbl)
REPORT_TABLES["Outlier (ditandai, tidak dibuang)"] = outlier_tbl

out_cols = [c for c in flags.columns if c.startswith("outlier_")]
any_out = flags[out_cols].any(axis=1)
print("Subjek dengan minimal satu flag outlier:", int(any_out.sum()))
display(df.loc[any_out, ["SUB_ID", "SITE_ID", "AGE_AT_SCAN", "FIQ"]]
        .assign(label=label[any_out].map(LABEL_NAMES)).join(flags.loc[any_out, out_cols]).head(40))
age_iqr_sites = df.loc[flags["outlier_age_iqr"], "SITE_ID"].value_counts()
if len(age_iqr_sites):
    note("Integritas", "temuan", f"{int(flags['outlier_age_iqr'].sum())} outlier umur (IQR global), terkonsentrasi di "
         f"{age_iqr_sites.head(5).to_dict()}. Umumnya mencerminkan rentang rekrutmen situs, bukan kesalahan data.")
note("Integritas", "keputusan", "Outlier hanya ditandai di audit_flags.csv dan tidak dibuang. Nilainya masih "
     "dalam rentang valid, jadi tidak ada bukti data salah.")

### CELL 3b — Verifikasi CELL 3

In [ ]:
assert n_dup_row == 0 and n_dup_sub == 0, "Ada duplikasi baris / SUB_ID"
assert n_dup_path == 0, "Satu file citra dipakai lebih dari satu subjek"
assert n_id_mismatch == 0, f"{n_id_mismatch} SUB_ID tidak cocok dengan nama file citra"
assert not only_table and not only_qc, "Himpunan SUB_ID tabel dan smri_qc (included) berbeda"
assert site_conflict == 0, "SITE_ID tabel dan smri_qc berbeda"
assert sex_vals <= SEX_VALID and hand_vals <= HANDEDNESS_VALID
assert range_tbl["n_di_luar_rentang"].sum() == 0, "Ada nilai di luar rentang valid"
assert df.equals(df_raw), "df berubah (harus read-only)"

print("Ringkasan integritas")
print(f"  baris={len(df)} | dtype tidak sesuai={len(bad_dtype)} | duplikat={n_dup_row + n_dup_sub + n_dup_path}")
print(f"  mismatch ID={n_id_mismatch} | di luar rentang={int(range_tbl['n_di_luar_rentang'].sum())}")
print(f"  flag outlier (≥1)={int(any_out.sum())} | QC: {flags['qc_decision'].value_counts().to_dict()}")
print("  NaN per kolom:", df.isna().sum()[lambda s: s > 0].to_dict())

### CELL 3c — Hash isi file citra dan `group_id`

**Tujuan:** membaca **isi** file citra setiap subjek (bukan hanya path) dan menghitung:
- `smri_file_sha256`: SHA-256 byte file `.nii.gz`;
- `smri_voxel_sha256`: SHA-256 array voxel terdekompresi + shape + dtype + affine.

Subjek dengan hash sama dikelompokkan ke satu `group_id` (subjek unik: `group_id` = `SUB_ID`-nya sendiri; subjek dengan citra identik: `group_id` = `SUB_ID` terkecil di kelompok). Pengelompokan memakai `GROUP_HASH` (default `voxel`), karena hash file bisa berbeda untuk citra yang sama (header gzip). Kelompok yang labelnya bertentangan atau lintas situs ditandai sebagai isu investigasi. **Tidak ada yang dihapus.**

Resume: hash disimpan ke `audit_cache/image_hashes.csv` dan dipakai ulang bila `smri_path`, ukuran, dan `mtime` file tidak berubah. File yang gagal dibaca dicatat di `hash_error` tanpa menghentikan proses.

**Output:** kolom `smri_file_sha256`, `smri_voxel_sha256`, `hash_error`, `group_id`, `group_size`, `flag_dup_group_label_conflict`, `flag_dup_group_cross_site` di `flags`.

In [ ]:
def voxel_sha256(path):
    img = nib.load(str(path))
    arr = np.ascontiguousarray(np.asanyarray(img.dataobj))
    h = hashlib.sha256(arr.tobytes())
    h.update(str(arr.shape).encode() + str(arr.dtype).encode() + np.round(img.affine, 4).tobytes())
    return h.hexdigest()


HASH_COLS = ["SUB_ID", "smri_path", "size", "mtime_ns", "file_sha256", "voxel_sha256", "hash_error"]
cache = (pd.read_csv(IMAGE_HASH_CACHE, dtype={"file_sha256": str, "voxel_sha256": str, "hash_error": str})
         .fillna({"file_sha256": "", "voxel_sha256": "", "hash_error": ""})
         if IMAGE_HASH_CACHE.exists() else pd.DataFrame(columns=HASH_COLS))
cache_idx = {(int(r.SUB_ID), r.smri_path, int(r.size), int(r.mtime_ns)): r
             for r in cache.itertuples() if not r.hash_error}

records, n_reused, n_computed, t0 = [], 0, 0, time.time()
for k, i in enumerate(df.index, start=1):
    sub_id, rel, path = int(df.at[i, "SUB_ID"]), str(df.at[i, "smri_path"]), smri_abs[i]
    rec = {"SUB_ID": sub_id, "smri_path": rel, "size": -1, "mtime_ns": -1,
           "file_sha256": "", "voxel_sha256": "", "hash_error": ""}
    try:
        st = path.stat()
        rec.update(size=st.st_size, mtime_ns=st.st_mtime_ns)
        hit = cache_idx.get((sub_id, rel, st.st_size, st.st_mtime_ns))
        if hit is not None:
            rec.update(file_sha256=hit.file_sha256, voxel_sha256=hit.voxel_sha256)
            n_reused += 1
        else:
            rec["file_sha256"] = sha256_file(path)
            rec["voxel_sha256"] = voxel_sha256(path)
            n_computed += 1
    except Exception as e:                         # dicatat, batch tetap jalan
        rec["hash_error"] = f"{type(e).__name__}: {e}"[:200]
    records.append(rec)
    if k % HASH_CACHE_EVERY == 0 or k == len(df):
        pd.DataFrame(records, columns=HASH_COLS).to_csv(IMAGE_HASH_CACHE, index=False)
        print(f"  {k}/{len(df)} | dipakai ulang={n_reused} | dihitung={n_computed} | {time.time() - t0:.0f} s")

hashes = pd.DataFrame(records, columns=HASH_COLS).set_index(df.index)
flags["smri_file_sha256"] = hashes["file_sha256"]
flags["smri_voxel_sha256"] = hashes["voxel_sha256"]
flags["hash_error"] = hashes["hash_error"]
n_hash_err = int(hashes["hash_error"].ne("").sum())

# group_id
key = hashes["voxel_sha256"] if GROUP_HASH == "voxel" else hashes["file_sha256"]
key = key.where(key.ne(""), "SUB_" + df["SUB_ID"].astype(str))    # gagal hash -> grup sendiri
flags["group_id"] = df["SUB_ID"].groupby(key.values).transform("min").astype(int).values
flags["group_size"] = flags.groupby("group_id")["SUB_ID"].transform("size")

# Kelompok >1 subjek
multi = flags["group_size"] > 1
grp = (flags.loc[multi, ["group_id", "SUB_ID", "SITE_ID"]]
       .assign(label=label[multi].map(LABEL_NAMES).values,
               AGE_AT_SCAN=df.loc[multi, "AGE_AT_SCAN"].values, FIQ=df.loc[multi, "FIQ"].values,
               file_sha256=flags.loc[multi, "smri_file_sha256"].str[:12],
               voxel_sha256=flags.loc[multi, "smri_voxel_sha256"].str[:12],
               qc_decision=flags.loc[multi, "qc_decision"])
       .sort_values(["group_id", "SUB_ID"]))
g_label_conf = label.groupby(flags["group_id"].values).transform("nunique").gt(1).values
g_site_conf = df["SITE_ID"].groupby(flags["group_id"].values).transform("nunique").gt(1).values
flags["flag_dup_group_label_conflict"] = g_label_conf
flags["flag_dup_group_cross_site"] = g_site_conf

# Perbandingan hash file vs hash voxel
n_multi_file = int(hashes.loc[hashes["file_sha256"].ne(""), "file_sha256"].duplicated(keep=False).sum())
n_multi_voxel = int(hashes.loc[hashes["voxel_sha256"].ne(""), "voxel_sha256"].duplicated(keep=False).sum())
print(f"\nSubjek dalam kelompok >1 menurut hash file : {n_multi_file}")
print(f"Subjek dalam kelompok >1 menurut hash voxel: {n_multi_voxel}")
print(f"Grup: {flags['group_id'].nunique()} | subjek: {len(flags)} | gagal hash: {n_hash_err}")

if multi.any():
    print("\nKelompok citra identik (>1 subjek):")
    display(grp)
    REPORT_TABLES["Kelompok citra identik (group_id)"] = grp
    summary = grp.groupby("group_id").agg(SUB_IDs=("SUB_ID", list), situs=("SITE_ID", lambda x: sorted(set(x))),
                                          label=("label", lambda x: sorted(set(x))),
                                          hash_file_sama=("file_sha256", lambda x: x.nunique() == 1))
    summary["label_bertentangan"] = summary["label"].map(len) > 1
    summary["lintas_situs"] = summary["situs"].map(len) > 1
    pheno_cols = ["AGE_AT_SCAN", "SEX", "FIQ", "HANDEDNESS_CATEGORY"]
    summary["fenotipik_berbeda"] = (df.loc[multi, pheno_cols].astype(str)
                                    .groupby(flags.loc[multi, "group_id"].values).nunique().gt(1).any(axis=1))
    display(summary)
    for gid, r in summary.iterrows():
        issues = [t for t, f in [("label bertentangan", r["label_bertentangan"]),
                                 ("lintas situs", r["lintas_situs"])] if f]
        kind = "tinjau" if (issues or r["fenotipik_berbeda"]) else "temuan"
        note("Citra identik", kind, f"group_id {gid}: SUB_ID {r['SUB_IDs']} ({', '.join(r['situs'])}; label "
             f"{', '.join(r['label'])}) memiliki citra identik"
             + ("" if r["hash_file_sama"] else " walau hash file .nii.gz berbeda")
             + (f". ISU INVESTIGASI: {', '.join(issues)}." if issues else
                (". Fenotipik berbeda: kemungkinan salah satu baris salah pasangan citra." if r["fenotipik_berbeda"]
                 else ". Fenotipik identik: kemungkinan entri ganda subjek yang sama.")))
    if n_multi_file < n_multi_voxel:
        note("Citra identik", "temuan", f"Hash file hanya menemukan {n_multi_file} subjek duplikat, hash voxel {n_multi_voxel}. "
             "group_id memakai hash voxel.")
    note("Citra identik", "keputusan", "Tidak ada subjek yang dihapus. Subjek satu group_id selalu ditempatkan di fold "
         "yang sama (StratifiedGroupKFold). Penghapusan, jika perlu, diputuskan di 05b dan dicatat di drop_log.csv.")
else:
    note("Citra identik", "temuan", "Tidak ada citra identik antar-SUB_ID.")
if n_hash_err:
    note("Citra identik", "tinjau", f"{n_hash_err} file gagal di-hash: "
         f"{flags.loc[flags['hash_error'].ne(''), ['SUB_ID', 'hash_error']].head(10).values.tolist()}")

### CELL 3d — Verifikasi CELL 3c

In [ ]:
assert len(hashes) == len(df) and flags["group_id"].notna().all()
assert ((hashes["file_sha256"].str.len() == 64) | hashes["hash_error"].ne("")).all(), "Ada subjek tanpa hash dan tanpa error"
assert (flags.loc[~multi, "group_id"] == flags.loc[~multi, "SUB_ID"]).all(), "Subjek unik harus punya group_id sendiri"
assert flags.groupby("group_id")["smri_voxel_sha256"].nunique().le(1).all() if GROUP_HASH == "voxel" else True
if REQUIRE_SMRI_ON_DISK:
    assert n_hash_err == 0, f"{n_hash_err} file gagal di-hash"
assert df.equals(df_raw)
print(f"Subjek: {len(flags)} | grup: {flags['group_id'].nunique()} | grup >1 subjek: {flags.loc[multi, 'group_id'].nunique()}")
print(f"Subjek di grup >1: {int(multi.sum())} | label bertentangan: {flags.loc[g_label_conf, 'group_id'].nunique()} grup | "
      f"lintas situs: {flags.loc[g_site_conf, 'group_id'].nunique()} grup")
print(f"Gagal hash: {n_hash_err} | dihitung: {n_computed} | dari cache: {n_reused}")
print(f"Cache: {IMAGE_HASH_CACHE}")

### CELL 4 — Audit missing value (fokus FIQ)

**Tujuan:**
- menghitung persentase NaN per kolom;
- untuk FIQ, membedakan NaN sungguhan (kosong di sumber) dari nilai sentinel `-9999` atau nilai tak terbaca/salah format, dengan melacak nilai mentah di `Phenotypic_V1_0b.csv`;
- melihat sebaran NaN per situs dan per label (tabel dan chi-square/Fisher);
- memeriksa konsistensi `FIQ_missing` dengan `FIQ.isna()`;
- menyimpulkan kemungkinan penyebab missing.

**Output:** tabel missing, tabel asal-usul NaN FIQ, uji asosiasi, flag `fiq_nan`, `fiq_raw_origin`, `handedness_nan` di `flags`.

In [ ]:
# 1) Persentase NaN per kolom
miss_tbl = pd.DataFrame({"n_NaN": df.isna().sum(), "pct_NaN": (df.isna().mean() * 100).round(2)})
display(miss_tbl)
REPORT_TABLES["Missing per kolom"] = miss_tbl.rename_axis("kolom")

fiq_nan = df["FIQ"].isna()
flags["fiq_nan"] = fiq_nan
flags["handedness_nan"] = df["HANDEDNESS_CATEGORY"].isna()

# 2) Asal-usul NaN FIQ dari data mentah
def classify_raw(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "SUB_ID tidak ada di data mentah"
    v = str(v).strip()
    if v == "":
        return "kosong di sumber"
    if v in RAW_MISSING_SENTINELS:
        return "sentinel -9999"
    try:
        x = float(v)
    except ValueError:
        return "tak terbaca / salah format"
    lo, hi = FIQ_VALID_RANGE
    return "numerik valid" if lo <= x <= hi else "numerik di luar rentang"

if raw_pheno is not None:
    raw_fiq = df["SUB_ID"].astype(str).map(raw_pheno["FIQ"])
    flags["fiq_raw_origin"] = raw_fiq.map(classify_raw)
    origin_tbl = pd.crosstab(flags["fiq_raw_origin"], fiq_nan.map({True: "FIQ NaN", False: "FIQ terisi"}))
    print("Asal-usul nilai FIQ (mentah) vs status di tabel:")
    display(origin_tbl)
    REPORT_TABLES["Asal-usul FIQ NaN"] = origin_tbl
    bad_origin = flags.loc[fiq_nan & flags["fiq_raw_origin"].eq("numerik valid")]
    if len(bad_origin):
        note("Missing", "tinjau", f"{len(bad_origin)} FIQ NaN padahal nilai mentah valid: {bad_origin['SUB_ID'].tolist()[:10]}")

    # Ketersediaan VIQ/PIQ dan jenis tes untuk subjek FIQ NaN
    def is_valid_num(v):
        return classify_raw(v) == "numerik valid"
    sub_nan = df.loc[fiq_nan, "SUB_ID"].astype(str)
    viq_ok = sub_nan.map(raw_pheno["VIQ"]).map(is_valid_num)
    piq_ok = sub_nan.map(raw_pheno["PIQ"]).map(is_valid_num)
    alt_tbl = pd.DataFrame({"SITE_ID": df.loc[fiq_nan, "SITE_ID"].values,
                            "VIQ&PIQ_tersedia": (viq_ok & piq_ok).values,
                            "FIQ_TEST_TYPE": sub_nan.map(raw_pheno["FIQ_TEST_TYPE"]).replace("", "(kosong)").values})
    alt_site = alt_tbl.groupby("SITE_ID").agg(n_FIQ_NaN=("SITE_ID", "size"),
                                              n_VIQ_PIQ_tersedia=("VIQ&PIQ_tersedia", "sum"),
                                              FIQ_TEST_TYPE=("FIQ_TEST_TYPE", lambda s: s.value_counts().to_dict()))
    print("Subjek FIQ NaN: ketersediaan VIQ/PIQ dan jenis tes per situs:")
    display(alt_site)
    REPORT_TABLES["FIQ NaN: VIQ/PIQ tersedia"] = alt_site
    n_alt = int(alt_tbl["VIQ&PIQ_tersedia"].sum())
    if n_alt:
        note("Missing", "tinjau", f"{n_alt} dari {int(fiq_nan.sum())} subjek FIQ NaN memiliki VIQ dan PIQ valid. "
             "FIQ tidak bisa dihitung sebagai rata-rata VIQ/PIQ tanpa tabel norma tes, jadi tidak dipakai otomatis. "
             "Putuskan di 05b apakah VIQ/PIQ boleh dipakai sebagai sumber informasi.")

# 3) Sebaran per situs dan per label
site_miss = (pd.DataFrame({"SITE_ID": df["SITE_ID"], "fiq_nan": fiq_nan, "hand_nan": flags["handedness_nan"]})
             .groupby("SITE_ID").agg(n=("fiq_nan", "size"), n_FIQ_NaN=("fiq_nan", "sum"),
                                     pct_FIQ_NaN=("fiq_nan", "mean"), n_HAND_NaN=("hand_nan", "sum")))
site_miss["pct_FIQ_NaN"] = (site_miss["pct_FIQ_NaN"] * 100).round(1)
site_miss = site_miss.sort_values("pct_FIQ_NaN", ascending=False)
display(site_miss[site_miss[["n_FIQ_NaN", "n_HAND_NaN"]].sum(axis=1) > 0])
REPORT_TABLES["Missing per situs (situs dengan NaN)"] = site_miss[site_miss[["n_FIQ_NaN", "n_HAND_NaN"]].sum(axis=1) > 0]

def chi2_report(ct, name):
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    low = float((expected < 5).mean() * 100)
    res = {"uji": name, "chi2": round(chi2, 3), "dof": dof, "p": p, "%sel_expected<5": round(low, 1)}
    if ct.shape == (2, 2):
        res["fisher_p"] = stats.fisher_exact(ct)[1]
    return res

ct_site = pd.crosstab(df["SITE_ID"], fiq_nan)
ct_label = pd.crosstab(label.map(LABEL_NAMES), fiq_nan)
display(pd.crosstab(label.map(LABEL_NAMES), fiq_nan.map({True: "FIQ NaN", False: "FIQ terisi"}), margins=True))
assoc_tbl = pd.DataFrame([chi2_report(ct_site, "FIQ NaN × SITE_ID"), chi2_report(ct_label, "FIQ NaN × label")])
hand_nan = flags["handedness_nan"]
if hand_nan.sum():
    assoc_tbl = pd.concat([assoc_tbl, pd.DataFrame([
        chi2_report(pd.crosstab(df["SITE_ID"], hand_nan), "HANDEDNESS NaN × SITE_ID"),
        chi2_report(pd.crosstab(label, hand_nan), "HANDEDNESS NaN × label")])], ignore_index=True)
display(assoc_tbl)
REPORT_TABLES["Uji asosiasi missing"] = assoc_tbl

# Label di situs yang FIQ-nya hilang (shortcut check)
struct_sites = site_miss.index[site_miss["pct_FIQ_NaN"] >= SITE_MISSING_STRUCTURAL * 100].tolist()
sporadic_sites = site_miss.index[(site_miss["n_FIQ_NaN"] > 0)
                                 & (site_miss["pct_FIQ_NaN"] < SITE_MISSING_STRUCTURAL * 100)].tolist()
print("Situs missing struktural:", struct_sites, "| situs missing sporadis:", sporadic_sites)
lab_in_nan = pd.crosstab(df.loc[fiq_nan, "SITE_ID"], label[fiq_nan].map(LABEL_NAMES))
display(lab_in_nan)

# 4) Konsistensi FIQ_missing
fiq_missing_mismatch = df["FIQ_missing"].astype(int).ne(fiq_nan.astype(int))
flags["flag_fiq_missing_inconsistent"] = fiq_missing_mismatch
print("FIQ_missing tidak konsisten dengan FIQ.isna():", int(fiq_missing_mismatch.sum()))
fiqm_vs_site = pd.Series(df["FIQ_missing"].values, index=df["SITE_ID"]).groupby(level=0).mean()
share_struct = float(df.loc[fiq_nan, "SITE_ID"].isin(struct_sites).mean()) if fiq_nan.any() else 0.0

# 5) Kesimpulan
p_site = assoc_tbl.loc[assoc_tbl["uji"] == "FIQ NaN × SITE_ID", "p"].iloc[0]
p_label = assoc_tbl.loc[assoc_tbl["uji"] == "FIQ NaN × label", "p"].iloc[0]
origin_counts = flags.loc[fiq_nan, "fiq_raw_origin"].value_counts().to_dict() if "fiq_raw_origin" in flags else {}
n_badformat = origin_counts.get("tak terbaca / salah format", 0)
note("Missing", "temuan", f"FIQ NaN = {int(fiq_nan.sum())} ({fiq_nan.mean():.1%}). Asal mentah: {origin_counts}. "
     + ("Tidak ada nilai salah format." if n_badformat == 0 else f"{n_badformat} nilai salah format (perlu ditinjau)."))
note("Missing", "temuan", f"Missing FIQ berasosiasi kuat dengan situs (p={p_site:.2e}). {share_struct:.0%} NaN berasal "
     f"dari situs dengan ≥{SITE_MISSING_STRUCTURAL:.0%} missing {struct_sites}, artinya FIQ tidak diukur/dilaporkan di "
     f"situs tersebut. Sisanya sporadis di {sporadic_sites}.")
note("Missing", "temuan", f"Asosiasi missing FIQ dengan label: p={p_label:.3f}.")
note("Missing", "tinjau", "FIQ_missing hampir identik dengan indikator situs "
     f"{struct_sites}. Jika dipakai sebagai fitur, ia bisa menjadi proksi SITE_ID (shortcut). Uji di 05b sebelum dipakai.")
note("Missing", "keputusan", "Tidak ada baris yang dibuang karena missing. Imputasi FIQ dilakukan di 05b, di-fit hanya "
     "pada data latih tiap fold.")

### CELL 4b — Verifikasi CELL 4

In [ ]:
assert int(fiq_missing_mismatch.sum()) == 0, "FIQ_missing tidak konsisten dengan FIQ.isna()"
assert int(fiq_nan.sum()) == int(df_raw["FIQ"].isna().sum())
assert df.equals(df_raw)
print(f"Baris: {len(df)} | FIQ NaN: {int(fiq_nan.sum())} | HANDEDNESS NaN: {int(hand_nan.sum())}")
print(f"FIQ_missing tidak konsisten: {int(fiq_missing_mismatch.sum())}")
print(f"Situs missing struktural: {struct_sites}")
print("Distribusi kelas pada subjek FIQ NaN:", label[fiq_nan].map(LABEL_NAMES).value_counts().to_dict())

### CELL 5 — Audit situs

**Tujuan:** profil tiap `SITE_ID` (n, n_ASD, n_TD, umur, FIQ, %FIQ_missing, %REVIEW, protokol citra dari `smri_qc.csv`), lalu menandai situs yang kemungkinan merupakan pecahan satu situs ABIDE I (standar 17 situs, terbaca 20). **Tidak ada penggabungan otomatis.**

**Output:** `site_tbl`, tabel kandidat situs terpecah, kolom `site_split_group` di `flags`.

In [ ]:
work = df.assign(label=label.values, review=flags["qc_decision"].eq("REVIEW").values)
site_tbl = work.groupby("SITE_ID").agg(
    n=("SUB_ID", "size"), n_ASD=("label", "sum"),
    age_mean=("AGE_AT_SCAN", "mean"), age_sd=("AGE_AT_SCAN", "std"),
    age_min=("AGE_AT_SCAN", "min"), age_max=("AGE_AT_SCAN", "max"),
    FIQ_mean=("FIQ", "mean"), pct_FIQ_missing=("FIQ_missing", "mean"),
    pct_female=("SEX", lambda s: (s == 2).mean()), pct_REVIEW=("review", "mean"))
site_tbl.insert(2, "n_TD", site_tbl["n"] - site_tbl["n_ASD"])
for c in ["pct_FIQ_missing", "pct_female", "pct_REVIEW"]:
    site_tbl[c] = (site_tbl[c] * 100).round(1)

# Protokol citra per situs (bentuk volume & voxel modus)
qc_site = qc_in.reindex(df["SUB_ID"]).assign(SITE_ID=df["SITE_ID"].values)
qc_site["voxel"] = qc_site[["voxel_x_mm", "voxel_y_mm", "voxel_z_mm"]].round(2).astype(str).agg("x".join, axis=1)
site_tbl["shape_modus"] = qc_site.groupby("SITE_ID")["shape"].agg(lambda s: s.mode().iloc[0])
site_tbl["voxel_modus_mm"] = qc_site.groupby("SITE_ID")["voxel"].agg(lambda s: s.mode().iloc[0])
site_tbl["n_protokol"] = qc_site.groupby("SITE_ID")[["shape", "voxel"]].apply(lambda g: len(g.drop_duplicates()))
site_tbl = site_tbl.round(2)
display(site_tbl)
REPORT_TABLES["Profil situs"] = site_tbl

# Kandidat situs terpecah (mis. UM_1/UM_2)
site_names = pd.Series(sorted(df["SITE_ID"].unique()))
base = site_names.map(lambda s: m.group("base") if (m := SITE_SPLIT_PATTERN.match(s)) else s)
split_groups = base.groupby(base).filter(lambda g: len(g) > 1)
site_to_group = dict(zip(site_names, base))
flags["site_split_group"] = df["SITE_ID"].map(lambda s: site_to_group[s] if s in set(site_names[split_groups.index]) else "")
n_sites = df["SITE_ID"].nunique()
print(f"Situs terbaca: {n_sites} | standar ABIDE I: {ABIDE_I_STANDARD_SITES} | basis unik setelah pola _<angka>: {base.nunique()}")
if len(split_groups):
    split_tbl = site_tbl.loc[site_names[split_groups.index],
                             ["n", "n_ASD", "n_TD", "age_mean", "FIQ_mean", "shape_modus", "voxel_modus_mm"]]
    split_tbl.insert(0, "grup", [site_to_group[s] for s in split_tbl.index])
    display(split_tbl)
    REPORT_TABLES["Kandidat situs terpecah"] = split_tbl
    note("Situs", "tinjau", f"{n_sites} SITE_ID vs {ABIDE_I_STANDARD_SITES} situs standar. Kandidat pecahan: "
         f"{split_tbl.groupby('grup').apply(lambda g: list(g.index)).to_dict()}. Di ABIDE I, sampel ini adalah "
         "kohort/sampel berbeda dari institusi yang sama, dan protokolnya bisa berbeda (lihat shape/voxel). "
         "Tidak digabung otomatis. Putuskan sebelum ComBat dan LOSO.")
note("Situs", "keputusan", "SITE_ID dipakai apa adanya (20 nilai) untuk stratifikasi dan LOSO.")
small_class = site_tbl.index[(site_tbl[["n_ASD", "n_TD"]].min(axis=1)) < LOSO_MIN_PER_CLASS].tolist()
if small_class:
    note("Situs", "temuan", f"Situs dengan < {LOSO_MIN_PER_CLASS} subjek pada salah satu kelas: {small_class}")

### CELL 5b — Verifikasi CELL 5

In [ ]:
assert site_tbl["n"].sum() == len(df)
assert site_tbl["n_ASD"].sum() == int(label.sum()) and site_tbl["n_TD"].sum() == int((label == 0).sum())
assert len(site_tbl) == n_sites
assert df.equals(df_raw)
print(f"Situs: {n_sites} | n min={site_tbl['n'].min()} ({site_tbl['n'].idxmin()}) | n max={site_tbl['n'].max()} ({site_tbl['n'].idxmax()})")
print(f"Kelas minimum per situs: {int(site_tbl[['n_ASD','n_TD']].min(axis=1).min())}")
print("Situs dengan >1 protokol citra:", site_tbl.index[site_tbl["n_protokol"] > 1].tolist())

### CELL 6 — Audit konfounding

**Tujuan:** menguji apakah umur, FIQ, dan SEX berbeda antara ASD dan TD, secara global dan per situs. Uji yang dipakai: Mann–Whitney U untuk variabel kontinu (effect size: rank-biserial *r*, positif = ASD lebih tinggi) dan chi-square/Fisher untuk SEX. p per situs dikoreksi Benjamini–Hochberg per variabel. FIQ diuji pada nilai yang tersedia saja.

**Output:** `conf_global`, `conf_site`.

In [ ]:
def bh_adjust(p):
    p = np.asarray(p, dtype=float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    if ok.sum() == 0:
        return out
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1].clip(max=1)
    res = np.empty_like(pv)
    res[order] = ranked
    out[ok] = res
    return out


def test_continuous(x, y, var):
    a, b = x[y == 1].dropna(), x[y == 0].dropna()
    row = {"variabel": var, "n_ASD": len(a), "n_TD": len(b),
           "ASD_mean": a.mean(), "TD_mean": b.mean(), "ASD_median": a.median(), "TD_median": b.median()}
    if len(a) >= 2 and len(b) >= 2:
        u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        row.update(uji="Mann-Whitney U", statistik=u, p=p, effect=2 * u / (len(a) * len(b)) - 1)
    else:
        row.update(uji="Mann-Whitney U", statistik=np.nan, p=np.nan, effect=np.nan)
    return row


def test_sex(sex, y):
    ct = pd.crosstab(y, sex).reindex(index=[1, 0], columns=[1, 2], fill_value=0)
    row = {"variabel": "SEX (% perempuan)", "n_ASD": int(ct.loc[1].sum()), "n_TD": int(ct.loc[0].sum()),
           "ASD_mean": ct.loc[1, 2] / max(ct.loc[1].sum(), 1) * 100, "TD_mean": ct.loc[0, 2] / max(ct.loc[0].sum(), 1) * 100}
    if (ct.sum(axis=0) > 0).all() and (ct.sum(axis=1) > 0).all():
        odds, p = stats.fisher_exact(ct.values)
        row.update(uji="Fisher exact", statistik=odds, p=p, effect=odds)   # effect = odds ratio
    else:
        row.update(uji="Fisher exact", statistik=np.nan, p=np.nan, effect=np.nan)
    return row


def confound_rows(sub, y):
    return [test_continuous(sub["AGE_AT_SCAN"], y, "AGE_AT_SCAN"),
            test_continuous(sub["FIQ"], y, "FIQ"),
            test_sex(sub["SEX"], y)]


conf_global = pd.DataFrame(confound_rows(df, label))
print("Konfounding global:")
display(conf_global.round(4).assign(p=conf_global["p"].map("{:.2e}".format)))
REPORT_TABLES["Konfounding global"] = conf_global

site_rows = []
for site, idx in df.groupby("SITE_ID").groups.items():
    for r in confound_rows(df.loc[idx], label.loc[idx]):
        site_rows.append({"SITE_ID": site, **r})
conf_site = pd.DataFrame(site_rows)
conf_site["p_BH"] = conf_site.groupby("variabel")["p"].transform(lambda s: pd.Series(bh_adjust(s.values), index=s.index))
conf_site["signifikan_BH"] = conf_site["p_BH"] < ALPHA
sig_site = conf_site[conf_site["signifikan_BH"]]
print(f"Per situs, signifikan setelah BH (α={ALPHA}):")
display(sig_site[["SITE_ID", "variabel", "n_ASD", "n_TD", "ASD_mean", "TD_mean", "effect", "p", "p_BH"]])
REPORT_TABLES["Konfounding per situs (signifikan BH)"] = sig_site[["SITE_ID", "variabel", "n_ASD", "n_TD",
                                                                   "ASD_mean", "TD_mean", "effect", "p_BH"]]

for _, r in conf_global.iterrows():
    sig = "berbeda signifikan" if r["p"] < ALPHA else "tidak berbeda signifikan"
    note("Konfounding", "temuan", f"{r['variabel']}: ASD={r['ASD_mean']:.2f} vs TD={r['TD_mean']:.2f}, "
         f"{sig} (p={r['p']:.2e}, effect={r['effect']:.3f}).")
if (conf_global.loc[conf_global["variabel"] == "FIQ", "p"] < ALPHA).any():
    note("Konfounding", "tinjau", "FIQ berbeda antara ASD dan TD. Perbedaan IQ sebagian merupakan karakteristik "
         "populasi ASD. Dokumentasikan bahwa model fenotipik dapat memanfaatkan sinyal ini, dan laporkan hasil "
         "dengan/tanpa FIQ (ablasi) di tahap pemodelan.")
note("Konfounding", "keputusan", "Konfounder hanya dilaporkan. Tidak ada matching atau pembuangan subjek di notebook ini.")

### CELL 6b — Verifikasi CELL 6

In [ ]:
assert len(conf_global) == 3
assert conf_site["SITE_ID"].nunique() == n_sites
assert conf_site["p_BH"].dropna().between(0, 1).all()
assert df.equals(df_raw)
print(f"Uji per situs: {len(conf_site)} | dapat diuji: {int(conf_site['p'].notna().sum())} | "
      f"signifikan BH: {int(conf_site['signifikan_BH'].sum())}")
print(conf_site.groupby("variabel")["signifikan_BH"].sum().to_dict())

### CELL 7 — Eksklusi, split dev/test, folds pada dev, daftar pilot

**Tujuan:**
1. **Eksklusi.** `EXCLUDE_SUB_IDS` dikeluarkan dari semua set pemodelan **hanya jika** bukti audit terkonfirmasi: satu `group_id`, hash voxel sama, fenotipik berbeda. Jika tidak terkonfirmasi, notebook berhenti. Dicatat di `exclusions.csv`.
2. **Held-out test.** Per strata `label × SITE_ID`, `round(TEST_FRACTION × n)` grup (`group_id`) dipilih acak (seed 42) sebagai test. Test tidak dipakai untuk fold, LOSO, tuning, atau eksperimen awal.
3. **Folds pada dev.** `fold_strat` dan `repeated_fold_1..3`: `StratifiedGroupKFold` 5-fold, strata `label × SITE_ID` (fallback `label_RARE` → label), `groups = group_id`. `loso_site` dan `loso_eligible` dihitung dari dev. Untuk baris test: kolom fold = -1, `loso_site` kosong, `loso_eligible = False`.
4. **Daftar pilot 05c.** `PILOT_N` subjek dari dev (seed 42), situs diacak lalu label berselang-seling.

**Output:** `folds` (1 baris per subjek non-eksklusi), `exclusions`, `pilot_ids` (ditulis di CELL 7b setelah verifikasi). Kolom folds: `SUB_ID, SITE_ID, label, group_id, split, fold_strat, repeated_fold_1..3, loso_site, loso_group_conflict, strata_fallback, loso_eligible`.

In [ ]:
# 1) Eksklusi berbasis bukti audit
pheno_cmp = ["AGE_AT_SCAN", "SEX", "FIQ", "HANDEDNESS_CATEGORY"]
ev = df.loc[df["SUB_ID"].isin(EXCLUDE_SUB_IDS), ["SUB_ID", "SITE_ID"] + pheno_cmp].merge(
    flags[["SUB_ID", "group_id", "group_size", "smri_voxel_sha256"]], on="SUB_ID")
missing_ex = sorted(set(EXCLUDE_SUB_IDS) - set(ev["SUB_ID"]))
assert not missing_ex, f"SUB_ID eksklusi tidak ada di data: {missing_ex}"
same_group = ev["group_id"].nunique() == 1 and (ev["group_size"] > 1).all()
same_voxel = ev["smri_voxel_sha256"].nunique() == 1 and ev["smri_voxel_sha256"].str.len().eq(64).all()
pheno_differs = bool((ev[pheno_cmp].astype(str).nunique() > 1).any())
print(f"Bukti eksklusi: satu group_id={same_group} | hash voxel sama={same_voxel} | fenotipik berbeda={pheno_differs}")
display(ev)
if not (same_group and same_voxel and pheno_differs):
    raise RuntimeError("Bukti eksklusi tidak terkonfirmasi oleh audit; subjek tidak dikeluarkan. Periksa EXCLUDE_SUB_IDS.")
evidence = (f"audit_flags: group_id={int(ev['group_id'].iloc[0])}, smri_voxel_sha256={ev['smri_voxel_sha256'].iloc[0][:16]}…; "
            "fenotipik berbeda: " + "; ".join(f"{c}={ev[c].tolist()}" for c in pheno_cmp if ev[c].astype(str).nunique() > 1))
exclusions = pd.DataFrame({"SUB_ID": ev["SUB_ID"].astype(int), "SITE_ID": ev["SITE_ID"], "step": "05a",
                           "reason": EXCLUDE_REASON, "evidence": evidence})
excluded = df["SUB_ID"].isin(EXCLUDE_SUB_IDS)
note("Split", "keputusan", f"Dikeluarkan dari semua set pemodelan: {sorted(EXCLUDE_SUB_IDS)} ({EXCLUDE_REASON}). "
     f"Bukti: {evidence}")

# 2) Held-out test per strata label × SITE_ID, di level group_id
pool = pd.DataFrame({"SUB_ID": df["SUB_ID"].values, "SITE_ID": df["SITE_ID"].values, "label": label.values,
                     "group_id": flags["group_id"].values})[~excluded.values]
grp = pool.groupby("group_id").agg(label=("label", "first"), SITE_ID=("SITE_ID", "first"), n=("SUB_ID", "size"),
                                   n_label=("label", "nunique"), n_site=("SITE_ID", "nunique"))
assert (grp["n_label"] == 1).all() and (grp["n_site"] == 1).all(), "group_id campur label/situs; split per strata tidak valid"
rng = np.random.default_rng(SEED)
test_groups = set()
for (lab, site), gg in grp.sort_index().groupby(["label", "SITE_ID"], sort=True):
    target = int(round(TEST_FRACTION * gg["n"].sum()))
    taken = 0
    for g in rng.permutation(gg.index.values):
        if taken >= target:
            break
        test_groups.add(g); taken += gg.at[g, "n"]
pool["split"] = np.where(pool["group_id"].isin(test_groups), "test", "dev")
dev_mask = pool["split"].eq("dev").values
dev = pool[dev_mask].reset_index(drop=True)
print(f"Split: dev {int(dev_mask.sum())} | test {int((~dev_mask).sum())} ({(~dev_mask).mean():.1%}) | dikeluarkan {int(excluded.sum())}")

# 3) Folds hanya pada dev
strata_full = dev["label"].astype(str) + "_" + dev["SITE_ID"]
counts = strata_full.map(strata_full.value_counts())
strata = strata_full.where(counts >= N_FOLDS, dev["label"].astype(str) + "_RARE")
rare_counts = strata.map(strata.value_counts())
strata = strata.where(rare_counts >= N_FOLDS, dev["label"].astype(str))
strata_fallback = strata != strata_full
print(f"Strata dev label×situs: {strata_full.nunique()} | min anggota: {strata_full.value_counts().min()} | "
      f"subjek fallback: {int(strata_fallback.sum())}")
groups = dev["group_id"].values


def make_folds(seed):
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold = np.full(len(dev), -1, dtype=int)
    for k, (_, val_idx) in enumerate(sgkf.split(np.zeros(len(dev)), strata, groups)):
        fold[val_idx] = k
    return fold


FOLD_COLS = ["fold_strat"] + [f"repeated_fold_{i}" for i in range(1, len(REPEAT_SEEDS) + 1)]
dev["fold_strat"] = make_folds(SEED)
for i, sd in enumerate(REPEAT_SEEDS, start=1):
    dev[f"repeated_fold_{i}"] = make_folds(sd)
group_site = dev.sort_values("SUB_ID").groupby("group_id")["SITE_ID"].first()
dev["loso_site"] = dev["group_id"].map(group_site)
dev["loso_group_conflict"] = dev.groupby("group_id")["SITE_ID"].transform("nunique").gt(1)
dev["strata_fallback"] = strata_fallback.values
site_class_min_dev = pd.crosstab(dev["SITE_ID"], dev["label"]).min(axis=1)
dev["loso_eligible"] = dev["loso_site"].map(site_class_min_dev >= LOSO_MIN_PER_CLASS).astype(bool)

test = pool[~dev_mask].copy()
for c in FOLD_COLS:
    test[c] = -1
test["loso_site"] = np.nan
test["loso_group_conflict"] = False
test["strata_fallback"] = False
test["loso_eligible"] = False

FOLDS_COLUMNS = (["SUB_ID", "SITE_ID", "label", "group_id", "split"] + FOLD_COLS
                 + ["loso_site", "loso_group_conflict", "strata_fallback", "loso_eligible"])
folds = pd.concat([dev, test], ignore_index=True)[FOLDS_COLUMNS].sort_values("SUB_ID").reset_index(drop=True)

# Tabel split
split_tbl = pd.crosstab(folds["SITE_ID"], [folds["split"], folds["label"].map({1: "ASD", 0: "TD"})])
split_tbl.columns = [f"{a}_{b}" for a, b in split_tbl.columns]
display(split_tbl)
REPORT_TABLES["Split dev/test per situs"] = split_tbl
loso_bad = sorted(set(dev.loc[~dev["loso_eligible"], "loso_site"]))
loso_bad_v1 = sorted(pd.crosstab(pool["SITE_ID"], pool["label"]).min(axis=1).loc[lambda x: x < LOSO_MIN_PER_CLASS].index)
note("Split", "keputusan", f"Held-out test {int((folds['split'] == 'test').sum())} subjek "
     f"({(folds['split'] == 'test').mean():.1%}), stratifikasi label×SITE_ID per group_id, seed {SEED}. "
     "Test hanya untuk evaluasi akhir; semua fold dan LOSO dari dev.")
note("Folds", "keputusan", f"fold_strat (dev): StratifiedGroupKFold {N_FOLDS}-fold, groups=group_id, strata label×SITE_ID, "
     f"seed {SEED}; repeated_fold_1..{len(REPEAT_SEEDS)} seed {REPEAT_SEEDS}. Fallback strata: {int(strata_fallback.sum())} subjek.")
note("Folds", "keputusan", f"LOSO (dev): {dev['loso_site'].nunique() - len(loso_bad)} situs eligible; tidak eligible "
     f"(< {LOSO_MIN_PER_CLASS} per kelas di dev): {loso_bad}.")
if set(loso_bad) - set(loso_bad_v1):
    note("Folds", "tinjau", f"Situs yang menjadi tidak eligible LOSO karena held-out test: {sorted(set(loso_bad) - set(loso_bad_v1))}. "
         "Ambang LOSO_MIN_PER_CLASS tidak diubah.")

# 4) Daftar pilot 05c dari dev
rng_p = np.random.default_rng(SEED)
dev_pool = folds[folds["split"].eq("dev")].merge(flags[["SUB_ID", "qc_decision"]], on="SUB_ID")
pilot = []
sites_p = list(rng_p.permutation(sorted(dev_pool["SITE_ID"].unique())))
lab_cycle = 0
while len(pilot) < PILOT_N:
    added = False
    for site in sites_p:
        if len(pilot) >= PILOT_N:
            break
        cand = dev_pool[(dev_pool["SITE_ID"] == site) & (dev_pool["label"] == lab_cycle % 2) & ~dev_pool["SUB_ID"].isin(pilot)
                        & ~dev_pool["group_id"].isin(dev_pool.loc[dev_pool["SUB_ID"].isin(pilot), "group_id"])]
        if len(cand):
            pilot.append(int(rng_p.choice(cand["SUB_ID"].values))); lab_cycle += 1; added = True
    assert added, "Kandidat pilot habis"
pilot_ids = (dev_pool[dev_pool["SUB_ID"].isin(pilot)][["SUB_ID", "SITE_ID", "label", "group_id", "fold_strat", "qc_decision"]]
             .sort_values("SUB_ID").reset_index(drop=True))
print(f"Pilot: {len(pilot_ids)} subjek | situs {pilot_ids['SITE_ID'].nunique()} | label {pilot_ids['label'].value_counts().to_dict()} | "
      f"QC {pilot_ids['qc_decision'].value_counts().to_dict()} | fold {pilot_ids['fold_strat'].value_counts().sort_index().to_dict()}")

### CELL 7b — Verifikasi split, folds, dan leakage, lalu tulis `folds.csv`, `exclusions.csv`, `pilot_ids.csv`

In [ ]:
is_dev, is_test = folds["split"].eq("dev"), folds["split"].eq("test")
# 1) Kelengkapan dan eksklusi
assert folds["SUB_ID"].is_unique and set(folds["SUB_ID"]) == set(df["SUB_ID"]) - set(EXCLUDE_SUB_IDS)
assert not folds["SUB_ID"].isin(EXCLUDE_SUB_IDS).any()
assert set(folds["split"]) == {"dev", "test"}
assert abs(is_test.mean() - TEST_FRACTION) < 0.02, f"Proporsi test {is_test.mean():.3f} jauh dari {TEST_FRACTION}"
# 2) Tidak ada group_id di dev dan test sekaligus
assert folds.groupby("group_id")["split"].nunique().eq(1).all(), "group_id terpecah antara dev dan test"
# 3) Test tidak punya fold/LOSO; dev lengkap
assert (folds.loc[is_test, FOLD_COLS] == -1).all().all() and folds.loc[is_test, "loso_site"].isna().all()
assert not folds.loc[is_test, "loso_eligible"].any()
assert folds.loc[is_dev, FOLD_COLS + ["loso_site"]].notna().all().all()
for c in FOLD_COLS:
    assert set(folds.loc[is_dev, c]) == set(range(N_FOLDS)), f"{c}: fold dev kosong / di luar 0..{N_FOLDS - 1}"
    assert (folds[is_dev].groupby(c)["label"].nunique() == 2).all(), f"{c}: fold dengan satu kelas"
# 4) Leakage: group_id dan hash voxel tepat di satu fold per skema (dev)
dv = folds[is_dev]
for c in FOLD_COLS + ["loso_site"]:
    assert (dv.groupby("group_id")[c].nunique() == 1).all(), f"LEAKAGE {c}: group_id di >1 fold"
vh = folds["SUB_ID"].map(dict(zip(flags["SUB_ID"], flags["smri_voxel_sha256"])))
for c in FOLD_COLS + ["split"]:
    m = vh.ne("")
    assert folds[m].groupby(vh[m])[c].nunique().le(1).all(), f"LEAKAGE {c}: citra identik lintas fold/split"
assert folds["label"].equals(folds["SUB_ID"].map(dict(zip(df["SUB_ID"], label))).astype(int))
# 5) Rasio label: test dan fold dev
global_ratio = folds["label"].mean()
print(f"Proporsi ASD: semua {global_ratio:.3f} | dev {folds.loc[is_dev, 'label'].mean():.3f} | test {folds.loc[is_test, 'label'].mean():.3f}")
assert abs(folds.loc[is_test, "label"].mean() - global_ratio) < 0.03
ratio_tbl = pd.DataFrame({c: dv.groupby(c)["label"].agg(["size", "mean"]).apply(
    lambda r: f"n={int(r['size'])}, ASD={r['mean']:.3f}", axis=1) for c in FOLD_COLS})
ratio_tbl.index.name = "fold (dev)"
display(ratio_tbl)
max_dev = max((dv.groupby(c)["label"].mean() - dv["label"].mean()).abs().max() for c in FOLD_COLS)
print(f"Deviasi maksimum proporsi ASD per fold dev: {max_dev:.3f}")
assert max_dev < 0.05
REPORT_TABLES["Rasio label per fold (dev)"] = ratio_tbl
site_fold = pd.crosstab(dv["SITE_ID"], dv["fold_strat"])
print("Situs dev yang tidak muncul di semua fold:", site_fold.index[(site_fold == 0).any(axis=1)].tolist() or "-")
print("Situs tanpa subjek test:", sorted(set(folds["SITE_ID"]) - set(folds.loc[is_test, "SITE_ID"])) or "-")
for c in FOLD_COLS[1:]:
    ari = adjusted_rand_score(dv["fold_strat"], dv[c])
    print(f"ARI fold_strat vs {c} (dev): {ari:.3f}")
    assert ari < 0.5
# 6) Pilot
assert len(pilot_ids) == PILOT_N and pilot_ids["SUB_ID"].is_unique
assert set(pilot_ids["SUB_ID"]) <= set(folds.loc[is_dev, "SUB_ID"]) and not pilot_ids["SUB_ID"].isin(EXCLUDE_SUB_IDS).any()

# Tulis
if FOLDS_PATH.exists() and not OLD_SCHEMA:
    old = pd.read_csv(FOLDS_PATH)
    same = old.shape == folds.shape and old.astype(str).equals(folds.astype(str))
    print("folds.csv (skema sama) sebelumnya:", "IDENTIK (deterministik)" if same else "BERBEDA, ditimpa")
folds.to_csv(FOLDS_PATH, index=False)
exclusions.to_csv(EXCLUSIONS_PATH, index=False)
pilot_ids.to_csv(PILOT_IDS_PATH, index=False)
print(f"Tersimpan: folds.csv {folds.shape} | exclusions.csv {exclusions.shape} | pilot_ids.csv {pilot_ids.shape}")
print(f"dev {int(is_dev.sum())} (ASD {int(folds.loc[is_dev, 'label'].sum())}) | test {int(is_test.sum())} "
      f"(ASD {int(folds.loc[is_test, 'label'].sum())}) | LOSO eligible (dev): {dv.loc[dv['loso_eligible'], 'loso_site'].nunique()}")

### CELL 8 — `audit_flags.csv`

**Tujuan:** menyimpan semua hasil audit per subjek di satu file terpisah (bukan di file sumber). File ini tidak berisi `label` dan tidak boleh dipakai sebagai fitur model.

**Output:** `audit_flags.csv`.

In [ ]:
bool_cols = [c for c in flags.columns if flags[c].dtype == bool]
flags["n_flags"] = flags[[c for c in bool_cols if c.startswith(("flag_", "outlier_"))]].sum(axis=1)
flags_out = flags.sort_values("SUB_ID").reset_index(drop=True)
assert "label" not in flags_out.columns and "DX_GROUP" not in flags_out.columns
assert flags_out["SUB_ID"].is_unique and len(flags_out) == len(df)
flags_out.to_csv(AUDIT_FLAGS_PATH, index=False)
print("Tersimpan:", AUDIT_FLAGS_PATH, flags_out.shape)
print("Jumlah True per flag:")
print(flags_out[bool_cols].sum().to_string())
print("Subjek dengan ≥1 flag:", int((flags_out["n_flags"] > 0).sum()))

### CELL 9 — `audit_report.md` dan verifikasi akhir

**Tujuan:** menulis laporan audit (temuan, keputusan, hal yang perlu ditinjau manual, tabel utama), lalu memastikan `cleaned_master_dataset.csv` tidak berubah (hash sebelum = sesudah).

**Output:** `audit_report.md`, ringkasan akhir.

In [ ]:
def md_table(frame, digits=3):
    frame = frame.copy()
    if not isinstance(frame.index, pd.RangeIndex):
        frame = frame.reset_index()
    def fmt(v):
        if isinstance(v, (float, np.floating)):
            if np.isnan(v):
                return ""
            return f"{v:.2e}" if (v != 0 and abs(v) < 1e-3) else f"{round(v, digits)}"
        return str(v).replace("|", "/")
    cols = [str(c) for c in frame.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join(lines)


find = pd.DataFrame(FINDINGS)
lines = ["# Audit Report — Notebook 05a", "",
         f"- Sumber: `{CLEANED_PATH.relative_to(PROJECT_ROOT).as_posix()}`",
         f"- SHA-256 sumber: `{HASH_BEFORE}`",
         f"- Subjek: {len(df)} (ASD {int(label.sum())}, TD {int((label == 0).sum())}), situs: {n_sites}",
         f"- Encoding DX_GROUP: {DX_SCHEME}; label: ASD = 1, TD = 0",
         f"- Seed: {SEED}; N_FOLDS: {N_FOLDS}; REPEAT_SEEDS: {REPEAT_SEEDS}; LOSO_MIN_PER_CLASS: {LOSO_MIN_PER_CLASS}",
         f"- Skema split: {SPLIT_SCHEMA_VERSION}; dikeluarkan {sorted(EXCLUDE_SUB_IDS)}; dev {int(is_dev.sum())}, "
         f"test {int(is_test.sum())} (TEST_FRACTION {TEST_FRACTION}); pilot 05c {PILOT_N} subjek dari dev",
         "- Notebook ini read-only: tidak ada baris dibuang, tidak ada nilai diubah, tidak ada transformasi di-fit.", ""]
for kind, title in [("temuan", "Temuan"), ("keputusan", "Keputusan"), ("tinjau", "Perlu ditinjau manual")]:
    lines += [f"## {title}", ""]
    sub = find[find["jenis"] == kind]
    lines += [f"- **[{r.bagian}]** {r.teks}" for r in sub.itertuples()] or ["- (tidak ada)"]
    lines.append("")
lines += ["## Tabel", ""]
for name, tbl in REPORT_TABLES.items():
    lines += [f"### {name}", "", md_table(tbl), ""]
AUDIT_REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print("Tersimpan:", AUDIT_REPORT_PATH, f"({len(lines)} baris)")

### CELL 9b — Verifikasi akhir

In [ ]:
HASH_AFTER = sha256_file(CLEANED_PATH)
print("SHA-256 sebelum :", HASH_BEFORE)
print("SHA-256 sesudah :", HASH_AFTER)
print("cleaned_master_dataset.csv tidak berubah:", HASH_BEFORE == HASH_AFTER)
assert HASH_BEFORE == HASH_AFTER, "cleaned_master_dataset.csv berubah!"
assert "label" not in pd.read_csv(CLEANED_PATH, nrows=1).columns
FOLDS_HASH_AFTER = sha256_file(FOLDS_PATH)
print("SHA-256 folds.csv sebelum:", FOLDS_HASH_BEFORE or "(belum ada)")
print("SHA-256 folds.csv sesudah:", FOLDS_HASH_AFTER)
if FOLDS_HASH_BEFORE is not None and not OLD_SCHEMA:
    print("folds.csv identik dengan sebelumnya:", FOLDS_HASH_BEFORE == FOLDS_HASH_AFTER)
    assert FOLDS_HASH_BEFORE == FOLDS_HASH_AFTER, "folds.csv berubah dari run sebelumnya!"
elif OLD_SCHEMA:
    print("folds.csv: skema lama diganti ke", SPLIT_SCHEMA_VERSION, "| versi lama diarsipkan:",
          (ARCHIVE_DIR / "folds.csv").relative_to(PROJECT_ROOT).as_posix())
    assert sha256_file(ARCHIVE_DIR / "folds.csv") == FOLDS_HASH_BEFORE, "Arsip folds.csv tidak sama dengan versi lama"

f_chk = pd.read_csv(FOLDS_PATH)
a_chk = pd.read_csv(AUDIT_FLAGS_PATH)
assert list(f_chk.columns) == FOLDS_COLUMNS
assert (f_chk[f_chk["split"] == "dev"].groupby("group_id")[FOLD_COLS + ["loso_site"]].nunique() == 1).all().all()
assert {"smri_file_sha256", "smri_voxel_sha256", "group_id"} <= set(a_chk.columns)
assert len(a_chk) == len(df_raw) and len(f_chk) == len(df_raw) - len(EXCLUDE_SUB_IDS)
assert set(a_chk["SUB_ID"]) == set(df_raw["SUB_ID"]) and set(f_chk["SUB_ID"]) == set(df_raw["SUB_ID"]) - set(EXCLUDE_SUB_IDS)
assert set(pd.read_csv(EXCLUSIONS_PATH)["SUB_ID"]) == set(EXCLUDE_SUB_IDS)
assert set(pd.read_csv(PILOT_IDS_PATH)["SUB_ID"]) <= set(f_chk.loc[f_chk["split"] == "dev", "SUB_ID"])
assert AUDIT_REPORT_PATH.stat().st_size > 0

print("\n=== RINGKASAN 05a ===")
print(f"Subjek          : {len(f_chk)} | ASD {int(f_chk['label'].sum())} | TD {int((f_chk['label'] == 0).sum())} | "
      f"dikeluarkan {len(EXCLUDE_SUB_IDS)}")
print(f"Split           : dev {int((f_chk['split'] == 'dev').sum())} | test {int((f_chk['split'] == 'test').sum())}")
print(f"Situs           : {f_chk['SITE_ID'].nunique()} | LOSO eligible: {f_chk.loc[f_chk['loso_eligible'], 'SITE_ID'].nunique()}")
print(f"Fold            : {N_FOLDS} × {len(FOLD_COLS)} skema | deviasi ASD maks: {max_dev:.3f}")
print(f"NaN di folds.csv: {int(f_chk.isna().sum().sum())}")
print(f"FIQ NaN         : {int(df_raw['FIQ'].isna().sum())} | HANDEDNESS NaN: {int(df_raw['HANDEDNESS_CATEGORY'].isna().sum())}")
print(f"File sMRI hilang: {n_smri_missing} | gagal hash: {n_hash_err}")
print(f"group_id        : {f_chk['group_id'].nunique()} | grup >1 subjek: {int(f_chk.groupby('group_id').size().gt(1).sum())} | "
      f"konflik LOSO: {int(f_chk['loso_group_conflict'].sum())}")
print(f"Temuan/keputusan/tinjau: {find['jenis'].value_counts().reindex(['temuan', 'keputusan', 'tinjau'], fill_value=0).to_dict()}")
print("Output:", [p.name for p in (FOLDS_PATH, EXCLUSIONS_PATH, PILOT_IDS_PATH, AUDIT_FLAGS_PATH, AUDIT_REPORT_PATH)])

### CELL 10 — Subset 20% (200 subjek dev)

**Tujuan:** daftar subjek untuk batch preprocessing 200 dan eksperimen subset. Tidak mengubah `folds.csv`.

**Aturan:**
1. Kandidat hanya `split == "dev"` (test tidak pernah masuk).
2. Alokasi dua tingkat dengan largest remainder (bagian bulat dulu, sisa kursi ke sisa pecahan terbesar;
   seri diputus urutan acak seed 42):
   a. kuota per `SITE_ID × label` = `SUBSET_N × n / n_dev` → proporsi ASD/TD per situs terjaga;
   b. kuota tiap `SITE_ID × label` dibagi ke `fold_strat` sebanding jumlah subjek di tiap fold.
3. Semua subjek pilot wajib masuk; kuota sel minimal = jumlah pilot di sel itu. Kelebihan kursi akibat aturan
   ini dikurangi dari sel non-pilot dengan sisa pecahan terkecil.
4. Di dalam sel `fold × situs × label`: pilot dulu, lalu sisanya dipilih acak (seed 42).

**Output:** `subset_20pct.csv` (`SUB_ID`, `SITE_ID`, `fold_strat`, `is_pilot`; tanpa label), `subset_20pct_rules.md`.

In [ ]:
dev_s = f_chk.loc[f_chk["split"] == "dev", ["SUB_ID", "SITE_ID", "label", "fold_strat"]].copy()
pilot_set = set(pd.read_csv(PILOT_IDS_PATH)["SUB_ID"].astype(int))
dev_s["stratum"] = dev_s["fold_strat"].astype(str) + "|" + dev_s["SITE_ID"] + "|" + dev_s["label"].astype(str)
dev_s["is_pilot"] = dev_s["SUB_ID"].isin(pilot_set)
rng_s = np.random.default_rng(SEED)


def allocate(tab, total):
    """Largest remainder dengan batas bawah n_pilot dan batas atas n. tab: kolom n, n_pilot."""
    t = tab.copy()
    t["tiebreak"] = rng_s.permutation(len(t))
    t["quota"] = t["n"] * total / t["n"].sum()
    t["take"] = np.maximum(np.floor(t["quota"]).astype(int), t["n_pilot"])
    t["frac"] = t["quota"] - np.floor(t["quota"])
    left = total - int(t["take"].sum())
    if left > 0:                                # tambah ke sisa pecahan terbesar
        order = t[t["take"] < t["n"]].sort_values(["frac", "tiebreak"], ascending=[False, True]).index
        for k in order[:left]:
            t.loc[k, "take"] += 1
    elif left < 0:                              # kurangi dari sel non-pilot dengan sisa pecahan terkecil
        order = t[t["take"] > t["n_pilot"]].sort_values(["frac", "tiebreak"], ascending=[True, True]).index
        for k in order[:-left]:
            t.loc[k, "take"] -= 1
    assert int(t["take"].sum()) == total and (t["take"] <= t["n"]).all() and (t["take"] >= t["n_pilot"]).all()
    return t


dev_s["site_label"] = dev_s["SITE_ID"] + "|" + dev_s["label"].astype(str)
lvl1 = allocate(dev_s.groupby("site_label").agg(n=("SUB_ID", "size"), n_pilot=("is_pilot", "sum")).sort_index(),
                SUBSET_N)
st = pd.concat([allocate(dev_s[dev_s["site_label"] == sl].groupby("stratum")
                         .agg(n=("SUB_ID", "size"), n_pilot=("is_pilot", "sum")).sort_index(), int(r["take"]))
                for sl, r in lvl1.iterrows()])
assert int(st["take"].sum()) == SUBSET_N

sel = []
for k, row in st.iterrows():
    g = dev_s[dev_s["stratum"] == k].sort_values("SUB_ID")
    pil = g[g["is_pilot"]]
    rest = g[~g["is_pilot"]]
    rest = rest.iloc[rng_s.permutation(len(rest))]
    sel.append(pd.concat([pil, rest.iloc[:int(row["take"]) - len(pil)]]))
subset = pd.concat(sel).sort_values("SUB_ID").reset_index(drop=True)
subset_out = subset[["SUB_ID", "SITE_ID", "fold_strat", "is_pilot"]]
subset_out.to_csv(SUBSET_PATH, index=False)

# ringkasan proporsi (subset vs dev)
cmp_lab = pd.DataFrame({"dev": dev_s["label"].value_counts(normalize=True), "subset": subset["label"].value_counts(normalize=True)})
cmp_fold = pd.DataFrame({"dev_n": dev_s["fold_strat"].value_counts(), "subset_n": subset["fold_strat"].value_counts(),
                         "dev_asd": dev_s.groupby("fold_strat")["label"].mean(),
                         "subset_asd": subset.groupby("fold_strat")["label"].mean()}).sort_index()
cmp_site = pd.DataFrame({"dev_n": dev_s["SITE_ID"].value_counts(), "subset_n": subset["SITE_ID"].value_counts(),
                         "expected": (dev_s["SITE_ID"].value_counts() * SUBSET_N / len(dev_s)).round(1),
                         "dev_asd": dev_s.groupby("SITE_ID")["label"].mean().round(3),
                         "subset_asd": subset.groupby("SITE_ID")["label"].mean().round(3)}).sort_index()
display(cmp_lab.round(4)); display(cmp_fold.round(3)); display(cmp_site)

SUBSET_HASH = sha256_file(SUBSET_PATH)
rules = [f"# subset_20pct.csv — aturan pembuatan", "",
         f"- Sumber: `folds.csv` (sha256 {sha256_file(FOLDS_PATH)}), `pilot_ids.csv` (sha256 {sha256_file(PILOT_IDS_PATH)})",
         f"- Hasil: `subset_20pct.csv` (sha256 {SUBSET_HASH}), {len(subset)} subjek, semua `split == dev`, seed {SEED}",
         "- Kandidat hanya dev; test held-out tidak pernah masuk.",
         "- Alokasi dua tingkat (largest remainder, seri diputus acak seed 42): kuota SITE_ID × label proporsional "
         "terhadap dev, lalu dibagi ke fold_strat sebanding jumlah subjek per fold.",
         f"- Semua {len(pilot_set)} subjek pilot wajib masuk (kuota strata minimal = jumlah pilot di strata).",
         "- Di dalam strata: pilot dulu, sisanya acak seed 42.",
         "- File tidak memuat label; label tetap hanya di folds.csv.", "",
         f"## Proporsi ASD: dev {dev_s['label'].mean():.4f} | subset {subset['label'].mean():.4f}", "",
         "## Per fold", "", md_table(cmp_fold.round(3).reset_index().rename(columns={"index": "fold_strat"})), "",
         "## Per situs", "", md_table(cmp_site.reset_index().rename(columns={"index": "SITE_ID"})), ""]
SUBSET_RULES_PATH.write_text("\n".join(rules), encoding="utf-8")
print("Tersimpan:", SUBSET_PATH.name, SUBSET_RULES_PATH.name)

### CELL 10b — Verifikasi subset

In [ ]:
chk = pd.read_csv(SUBSET_PATH)
dev_ids = set(f_chk.loc[f_chk["split"] == "dev", "SUB_ID"])
assert len(chk) == SUBSET_N and chk["SUB_ID"].is_unique
assert set(chk["SUB_ID"]) <= dev_ids, "subset memuat subjek non-dev"
assert pilot_set <= set(chk["SUB_ID"]), "tidak semua pilot masuk subset"
assert not set(chk["SUB_ID"]) & set(EXCLUDE_SUB_IDS)
assert "label" not in chk.columns
assert set(chk["SITE_ID"]) == set(dev_s["SITE_ID"]) and set(chk["fold_strat"]) == set(dev_s["fold_strat"])
asd_gap = abs(subset["label"].mean() - dev_s["label"].mean())
fold_gap = (cmp_fold["subset_asd"] - cmp_fold["dev_asd"]).abs().max()
site_gap = (cmp_site["subset_n"] - cmp_site["expected"]).abs().max()
assert asd_gap < 0.02, f"proporsi ASD subset menyimpang {asd_gap:.3f}"
assert sha256_file(FOLDS_PATH) == FOLDS_HASH_AFTER, "folds.csv berubah setelah CELL 10"
print(f"Subset: {len(chk)} subjek dev | pilot {int(chk['is_pilot'].sum())} | situs {chk['SITE_ID'].nunique()} | "
      f"fold {chk['fold_strat'].nunique()}")
print(f"Proporsi ASD dev {dev_s['label'].mean():.4f} vs subset {subset['label'].mean():.4f} (selisih {asd_gap:.4f}); "
      f"selisih ASD per fold maks {fold_gap:.3f}; selisih n per situs vs harapan maks {site_gap:.1f}")
print("folds.csv tetap:", sha256_file(FOLDS_PATH))
print("subset_20pct.csv:", SUBSET_HASH)